# 00 - Load and Clean
**QSS 20 Final Project | Olivia Tak**

This notebook loads the raw YouTube trending dataset, runs initial inspection, filters to 2023, deduplicates by video ID, computes days on trending, log-transforms virality metrics, and maps category labels.

**Inputs:** `../data/US_youtube_trending_data.csv`, `../data/US_category_id.json`  
**Outputs:** `../data/df_2023_clean.csv`

## Imports

In [1]:
import pandas as pd
import numpy as np
import json
from IPython.display import display

## Helper functions

In [2]:
def add_log_columns(df, cols):
    """Log-transform numeric columns to address right skew.

    Adds a log_{col} column for each col in cols.
    Uses log(x + 1) to handle zero values.

    Parameters
    ----------
    df   : DataFrame
    cols : list of str, column names to transform

    Returns
    -------
    DataFrame with new log columns added in place
    """
    for col in cols:
        df[f'log_{col}'] = np.log(df[col] + 1)
    return df


def map_categories(df, json_path, id_col='categoryId'):
    """Load YouTube category labels from JSON and map onto df.

    Parameters
    ----------
    df        : DataFrame
    json_path : str, path to US_category_id.json
    id_col    : str, column in df containing numeric category IDs

    Returns
    -------
    DataFrame with 'category' column added
    """
    with open(json_path, 'r') as f:
        category_data = json.load(f)
    category_dict = {
        int(item['id']): item['snippet']['title']
        for item in category_data['items']
    }
    df['category'] = df[id_col].map(category_dict)
    return df

## Load raw data

In [3]:
df = pd.read_csv('../data/US_youtube_trending_data.csv')
df['trending_date'] = pd.to_datetime(df['trending_date'])
print(f'Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')

Loaded: 268,787 rows, 16 columns


## Initial inspection

In [4]:
display(df.head())

,video_id,title,publishedAt,channelId,channelTitle,categoryId,trending_date,tags,view_count,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,description
0,3C66w5Z0ixs,I ASKED HER TO BE MY GIRLFRIEND...,2020-08-11T19:20:14Z,UCvtRTOMP2TqYqu51xNrqAzg,Brawadis,22,2020-08-12 00:00:00+00:00,brawadis|prank|basketball|skits|ghost|funny vi...,1514614,156908,5855,35313,https://i.ytimg.com/vi/3C66w5Z0ixs/default.jpg,False,False,SUBSCRIBE to BRAWADIS ▶ http://bit.ly/Subscrib...
1,M9Pmf9AB4Mo,Apex Legends | Stories from the Outlands – “Th...,2020-08-11T17:00:10Z,UC0ZV6M2THA81QT9hrVWJG3A,Apex Legends,20,2020-08-12 00:00:00+00:00,Apex Legends|Apex Legends characters|new Apex ...,2381688,146739,2794,16549,https://i.ytimg.com/vi/M9Pmf9AB4Mo/default.jpg,False,False,"While running her own modding shop, Ramya Pare..."
2,J78aPJ3VyNs,I left youtube for a month and THIS is what ha...,2020-08-11T16:34:06Z,UCYzPXprvl5Y-Sf0g4vX-m6g,jacksepticeye,24,2020-08-12 00:00:00+00:00,jacksepticeye|funny|funny meme|memes|jacksepti...,2038853,353787,2628,40221,https://i.ytimg.com/vi/J78aPJ3VyNs/default.jpg,False,False,I left youtube for a month and this is what ha...
3,kXLn3HkpjaA,XXL 2020 Freshman Class Revealed - Official An...,2020-08-11T16:38:55Z,UCbg_UMjlHJg_19SZckaKajg,XXL,10,2020-08-12 00:00:00+00:00,xxl freshman|xxl freshmen|2020 xxl freshman|20...,496771,23251,1856,7647,https://i.ytimg.com/vi/kXLn3HkpjaA/default.jpg,False,False,Subscribe to XXL → http://bit.ly/subscribe-xxl...
4,VIUo6yapDbc,Ultimate DIY Home Movie Theater for The LaBran...,2020-08-11T15:10:05Z,UCDVPcEbVLQgLZX0Rt6jo34A,Mr. Kate,26,2020-08-12 00:00:00+00:00,The LaBrant Family|DIY|Interior Design|Makeove...,1123889,45802,964,2196,https://i.ytimg.com/vi/VIUo6yapDbc/default.jpg,False,False,Transforming The LaBrant Family's empty white ...


In [5]:
# check for missing values
print('Missing values per column:')
display(
    df.isnull().sum()
      .reset_index()
      .rename(columns={'index': 'column', 0: 'n_missing'})
      .query('n_missing > 0')
)

Missing values per column:


,column,n_missing
15,description,4549


In [6]:
# full date range of the raw dataset
print(f'Date range: {df["trending_date"].min().date()} to {df["trending_date"].max().date()}')

Date range: 2020-08-12 to 2024-04-15


## Filter to 2023

The raw dataset spans 2020-2024. We focus on 2023 only for a consistent, recent time window.

In [7]:
n_before = len(df)
df_2023 = df[df['trending_date'].dt.year == 2023].copy()
n_after = len(df_2023)
print(f'Before filter: {n_before:,} rows')
print(f'After filter to 2023: {n_after:,} rows')
print(f'Dropped: {n_before - n_after:,} rows')

Before filter: 268,787 rows
After filter to 2023: 72,397 rows
Dropped: 196,390 rows


## Count days trending and deduplicate

Each video can appear on the trending page multiple days, creating duplicate rows. We count how many days each video trended, then deduplicate to one row per video keeping its first trending appearance.

In [8]:
# count trending appearances per video before deduplication
days_trending = (
    df_2023.groupby('video_id')['trending_date']
           .count()
           .reset_index()
           .rename(columns={'trending_date': 'days_trending'})
)
print('Days trending distribution:')
display(days_trending['days_trending'].describe().round(2).reset_index().rename(columns={'index': 'statistic', 'days_trending': 'value'}))

Days trending distribution:


,statistic,value
0,count,12176.00
1,mean,5.95
2,std,1.87
3,min,1.00
4,25%,5.00
5,50%,6.00
6,75%,7.00
7,max,37.00


In [9]:
# deduplicate: keep first trending appearance per video
n_before_dedup = len(df_2023)
df_2023 = df_2023.sort_values('trending_date').drop_duplicates(subset='video_id', keep='first')
print(f'Before deduplication: {n_before_dedup:,} rows')
print(f'After deduplication:  {len(df_2023):,} rows')
print(f'Unique videos retained: {df_2023["video_id"].nunique():,}')

Before deduplication: 72,397 rows
After deduplication:  12,176 rows
Unique videos retained: 12,176


In [10]:
# merge days_trending count back onto deduplicated dataframe (left join on video_id)
n_before_merge = len(df_2023)
df_2023 = df_2023.merge(days_trending, on='video_id', how='left')
n_after_merge = len(df_2023)
print(f'Before merge: {n_before_merge:,} rows')
print(f'After merge:  {n_after_merge:,} rows')
assert n_before_merge == n_after_merge, 'Row count changed after merge -- check for duplicates in days_trending'

Before merge: 12,176 rows
After merge:  12,176 rows


## Log-transform virality metrics

View counts, likes, and comment counts are heavily right-skewed. Log transformation produces a more symmetric distribution and is standard practice before running t-tests and regression.

In [11]:
df_2023 = add_log_columns(df_2023, cols=['view_count', 'likes', 'comment_count'])
print(f'Log columns added: log_view_count, log_likes, log_comment_count')

Log columns added: log_view_count, log_likes, log_comment_count


## Map category labels

In [12]:
df_2023 = map_categories(df_2023, '../data/US_category_id.json')
print(f'Categories mapped. Unique categories: {df_2023["category"].nunique()}')

Categories mapped. Unique categories: 15


## Save cleaned data

In [13]:
# rename log columns to consistent names used in downstream notebooks
df_2023 = df_2023.rename(columns={
    'log_view_count':  'log_views',
    'log_comment_count': 'log_comments'
})

df_2023.to_csv('../data/df_2023_clean.csv', index=False)
print(f'Saved df_2023_clean.csv: {df_2023.shape[0]:,} rows, {df_2023.shape[1]} columns')
print(f'Columns: {df_2023.columns.tolist()}')

Saved df_2023_clean.csv: 12,176 rows, 21 columns
Columns: ['video_id', 'title', 'publishedAt', 'channelId', 'channelTitle', 'categoryId', 'trending_date', 'tags', 'view_count', 'likes', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'description', 'days_trending', 'log_views', 'log_likes', 'log_comments', 'category']
